In [1]:
import numpy as np
import pandas as pd
import textwrap
import nltk
from nltk import word_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer

In [2]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [3]:
data = pd.read_csv('/content/bbc-text.csv')

In [4]:
data.head()

,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...
3,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,ocean s twelve raids box office ocean s twelve...


In [5]:
labels = set(data['category'])
labels

{'business', 'entertainment', 'politics', 'sport', 'tech'}

In [6]:
label = 'business'

In [8]:
texts = data[data['category'] == label]['text']
texts

1      worldcom boss  left books alone  former worldc...
11     virgin blue shares plummet 20% shares in austr...
12     crude oil prices back above $50 cold weather a...
15     s korean credit card firm rescued south korea ...
18     japanese banking battle at an end japan s sumi...
                             ...                        
870    macy s owner buys rival for $11bn us retail gi...
879    singapore growth at 8.1% in 2004 singapore s e...
895    jp morgan admits us slavery links thousands of...
896    troubled marsh under sec scrutiny the us stock...
904    marsh executive in guilty plea an executive at...
Name: text, Length: 210, dtype: object

In [9]:
# collect counts

probs= {}
for doc in texts:
  lines = doc.split('\n')
  for line in lines:
    tokens = word_tokenize(line)
    for i in range(len(tokens) - 2):
      t_0 = tokens[i]
      t_1 = tokens[i+1]
      t_2 = tokens[i+2]
      key = (t_0, t_2)
      if key not in probs:
        probs[key] = {}

      if t_1 not in probs[key]:
        probs[key][t_1] = 1
      else:
        probs[key][t_1] +=1



In [10]:
# Normalize Probabilities

for key, d in probs.items():
  total = sum(d.values())
  for k,v in d.items():
    d[k] = v/ total

In [ ]:
probs

In [12]:
texts.iloc[0].split('\n')

['worldcom boss  left books alone  former worldcom boss bernie ebbers  who is accused of overseeing an $11bn (£5.8bn) fraud  never made accounting decisions  a witness has told jurors.  david myers made the comments under questioning by defence lawyers who have been arguing that mr ebbers was not responsible for worldcom s problems. the phone company collapsed in 2002 and prosecutors claim that losses were hidden to protect the firm s shares. mr myers has already pleaded guilty to fraud and is assisting prosecutors.  on monday  defence lawyer reid weingarten tried to distance his client from the allegations. during cross examination  he asked mr myers if he ever knew mr ebbers  make an accounting decision  .  not that i am aware of   mr myers replied.  did you ever know mr ebbers to make an accounting entry into worldcom books   mr weingarten pressed.  no   replied the witness. mr myers has admitted that he ordered false accounting entries at the request of former worldcom chief financ

In [13]:
def spin_document(doc):
  # split the documents into lines

  lines = doc.split('\n')
  output = []
  for line in lines:
    if line:
      new_line = spin_line(line)
    else:
      new_line = line
    output.append(new_line)
  return "\n".join(output)

In [ ]:
probs

In [19]:
detokenizer = TreebankWordDetokenizer()

In [21]:
texts

1      worldcom boss  left books alone  former worldc...
11     virgin blue shares plummet 20% shares in austr...
12     crude oil prices back above $50 cold weather a...
15     s korean credit card firm rescued south korea ...
18     japanese banking battle at an end japan s sumi...
                             ...                        
870    macy s owner buys rival for $11bn us retail gi...
879    singapore growth at 8.1% in 2004 singapore s e...
895    jp morgan admits us slavery links thousands of...
896    troubled marsh under sec scrutiny the us stock...
904    marsh executive in guilty plea an executive at...
Name: text, Length: 210, dtype: object

In [23]:
# Example code with error handling
text_row = texts.iloc[0]
lines = text_row.split("\n")

if len(lines) > 2:
    third_line = lines[2]
    print("Third line:", third_line)
else:
    print("Text row does not contain at least three lines.")


Text row does not contain at least three lines.


In [18]:
detokenizer.detokenize(word_tokenize(texts.iloc[0].split("\n")[2]))

IndexError: list index out of range

In [24]:
def sample_word(d):
  p0= np.random.random()
  cumulative = 0
  for t, p in d.items():
    cumulative += p
    if p0 < cumulative:
      return t
  assert(False)


In [26]:
def spin_line(line):
  tokens = word_tokenize(line)
  i = 0
  output = [tokens[0]]
  while i < (len(tokens)-2):
    t_0 = tokens[i]
    t_1 = tokens[i+1]
    t_2 = tokens[i+2]
    key = (t_0, t_2)
    p_dist = probs[key]
    if len(p_dist) > 1 and np.random.random() < 0.3:
      middle = sample_word(p_dist)
      output.append(t_1)
      output.append("<" + middle + ">")
      output.append(t_2)

      i +=2
    else:
      output.append(t_1)
      i+=1
  if i == len(tokens) -2:
    output.append(tokens[-1])
  return detokenizer.detokenize(output)



In [27]:
np.random.seed(42)

In [28]:
i = np.random.choice(texts.shape[0])
doc = texts.iloc[i]
new_doc = spin_document(doc)

In [30]:
print(textwrap.fill(
    new_doc, replace_whitespace = False,
    fix_sentence_endings = True
))

bush to get tough on deficit <deficit> us president george w bush has
pledged to introduce a tough federal budget next february <year> in a
bid <proposal> to halve <reject> the country s deficit <admission> in
five years . <because> the us budget and its <its> trade deficit are
both deep in the red helping <helping> to push the dollar to lows
against the euro and fuelling fears about the economy <investigation>.
mr bush indicated there would be strict discipline on non-defence
<non-defence> spending in <in> the budget . the vow <decision> to cut
<promote> the deficit had been one <one> of his re-election
declarations . the <after> federal budget deficit hit a record $412bn
(£211.6bn) in the 12 months to 30 september and $377bn in the previous
year . we will submit a budget that fits the times mr bush <mandelson>
said . <that> it will provide every tool and resource to the military
<agreement> will protect <accept> the homeland and meet other
priorities of the government . the us <firm>